In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [2]:
df = pd.read_csv("Hindi_English_Truncated_Corpus.csv")

df = df[['english_sentence', 'hindi_sentence']].dropna()

df = df.drop_duplicates()

print(df.shape)
df.head()

(124825, 2)


,english_sentence,hindi_sentence
0,politicians do not have permission to do what ...,"राजनीतिज्ञों के पास जो कार्य करना चाहिए, वह कर..."
1,"I'd like to tell you about one such child,",मई आपको ऐसे ही एक बच्चे के बारे में बताना चाहू...
2,This percentage is even greater than the perce...,यह प्रतिशत भारत में हिन्दुओं प्रतिशत से अधिक है।
3,what we really mean is that they're bad at not...,हम ये नहीं कहना चाहते कि वो ध्यान नहीं दे पाते
4,.The ending portion of these Vedas is called U...,इन्हीं वेदों का अंतिम भाग उपनिषद कहलाता है।


In [3]:
df = df.iloc[:30000]
print(df.shape)

(30000, 2)


In [4]:
df['english_sentence'] = df['english_sentence'].astype(str).str.lower()
df['hindi_sentence'] = df['hindi_sentence'].astype(str)

df['hindi_sentence'] = (
    '<start> ' + 
    df['hindi_sentence'] + 
    ' <end>'
)

In [5]:
eng_tokenizer = Tokenizer(
    num_words=20000,
    filters=''
)

eng_tokenizer.fit_on_texts(df['english_sentence'])

eng_sequences = eng_tokenizer.texts_to_sequences(
    df['english_sentence']
)

eng_vocab_size = len(eng_tokenizer.word_index) + 1

print("English vocabulary:", eng_vocab_size)

English vocabulary: 45296


In [6]:
hin_tokenizer = Tokenizer(
    num_words=30000,
    filters=''
)

hin_tokenizer.fit_on_texts(df['hindi_sentence'])

hin_sequences = hin_tokenizer.texts_to_sequences(
    df['hindi_sentence']
)

hin_vocab_size = len(hin_tokenizer.word_index) + 1

print("Hindi vocabulary:", hin_vocab_size)

Hindi vocabulary: 46076


In [7]:
max_eng_len = max(len(seq) for seq in eng_sequences)
max_hin_len = max(len(seq) for seq in hin_sequences)

print("Maximum English length:", max_eng_len)
print("Maximum Hindi length:", max_hin_len)

Maximum English length: 206
Maximum Hindi length: 307


In [8]:
max_eng_len = min(max_eng_len, 30)
max_hin_len = min(max_hin_len, 30)

print(max_eng_len)
print(max_hin_len)

30
30


In [9]:
encoder_input_data = pad_sequences(
    eng_sequences,
    maxlen=max_eng_len,
    padding='post'
)

decoder_sequences = pad_sequences(
    hin_sequences,
    maxlen=max_hin_len,
    padding='post'
)

In [10]:
decoder_input_data = decoder_sequences[:, :-1]

decoder_target_data = decoder_sequences[:, 1:]

In [11]:
embedding_dim = 256
latent_dim = 256

In [12]:
encoder_inputs = Input(
    shape=(None,),
    name="encoder_input"
)

encoder_embedding = Embedding(
    input_dim=eng_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True
)(encoder_inputs)

encoder_lstm = LSTM(
    latent_dim,
    return_state=True
)

encoder_outputs, state_h, state_c = encoder_lstm(
    encoder_embedding
)

encoder_states = [state_h, state_c]

In [13]:
decoder_inputs = Input(
    shape=(None,),
    name="decoder_input"
)
decoder_embedding_layer = Embedding(
    input_dim=hin_vocab_size,
    output_dim=embedding_dim,
    mask_zero=True
)

decoder_embedding = decoder_embedding_layer(
    decoder_inputs
)
decoder_lstm = LSTM(
    latent_dim,
    return_sequences=True,
    return_state=True
)

decoder_outputs, _, _ = decoder_lstm(
    decoder_embedding,
    initial_state=encoder_states
)

In [14]:
decoder_dense = Dense(
    hin_vocab_size,
    activation='softmax'
)

decoder_outputs = decoder_dense(
    decoder_outputs
)

In [15]:
model = Model(
    [encoder_inputs, decoder_inputs],
    decoder_outputs
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ encoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ decoder_input       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 256) │ 11,595,776 │ encoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ not_equal           │ (None, None)      │          0 │ encoder_input[0]… │
│ (NotEqual)          │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, None, 256) │ 11,795,456 │ decoder_input[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 256),     │    525,312 │ embedding[0][0],  │
│                     │ (None, 256),      │            │ not_equal[0][0]   │
│                     │ (None, 256)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ [(None, None,     │    525,312 │ embedding_1[0][0… │
│                     │ 256), (None,      │            │ lstm[0][1],       │
│                     │ 256), (None,      │            │ lstm[0][2]        │
│                     │ 256)]             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None,      │ 11,841,532 │ lstm_1[0][0]      │
│                     │ 46076)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 36,283,388 (138.41 MB)

 Trainable params: 36,283,388 (138.41 MB)

 Non-trainable params: 0 (0.00 B)

In [16]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [17]:
from sklearn.model_selection import train_test_split

(
    encoder_train,
    encoder_test,
    decoder_train,
    decoder_test,
    target_train,
    target_test
) = train_test_split(
    encoder_input_data,
    decoder_input_data,
    decoder_target_data,
    test_size=0.1,
    random_state=42
)

In [18]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

In [19]:
history = model.fit(
    [encoder_train, decoder_train],
    target_train,
    validation_data=(
        [encoder_test, decoder_test],
        target_test
    ),
    batch_size=64,
    epochs=5,
    callbacks=[early_stop]
)

Epoch 1/5
422/422 ━━━━━━━━━━━━━━━━━━━━ 1167s 3s/step - accuracy: 0.1258 - loss: 6.8348 - val_accuracy: 0.1734 - val_loss: 6.2259
Epoch 2/5
 92/422 ━━━━━━━━━━━━━━━━━━━━ 14:56 3s/step - accuracy: 0.1796 - loss: 6.0143

KeyboardInterrupt: 

In [ ]:
loss, accuracy = model.evaluate(
    [encoder_test, decoder_test],
    target_test
)

print("Test Loss:", loss)
print("Test Accuracy:", accuracy)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])

plt.xlabel("Epoch")
plt.ylabel("Accuracy")

plt.legend([
    "Training Accuracy",
    "Validation Accuracy"
])

plt.show()

In [ ]:
encoder_model = Model(
    encoder_inputs,
    encoder_states
)
decoder_state_input_h = Input(
    shape=(latent_dim,)
)

decoder_state_input_c = Input(
    shape=(latent_dim,)
)

decoder_states_inputs = [
    decoder_state_input_h,
    decoder_state_input_c
]
decoder_emb = decoder_embedding_layer(
    decoder_inputs
)
decoder_outputs, state_h, state_c = decoder_lstm(
    decoder_emb,
    initial_state=decoder_states_inputs
)
decoder_states = [
    state_h,
    state_c
]

decoder_outputs = decoder_dense(
    decoder_outputs
)

def translate_sentence(sentence):

    sentence = sentence.lower()

    sequence = eng_tokenizer.texts_to_sequences(
        [sentence]
    )

    sequence = pad_sequences(
        sequence,
        maxlen=max_eng_len,
        padding='post'
    )

    states_value = encoder_model.predict(
        sequence,
        verbose=0
    )

    start_token = hin_tokenizer.word_index['<start>']
    end_token = hin_tokenizer.word_index['<end>']

    target_seq = np.array([[start_token]])

    decoded_sentence = []

    for _ in range(max_hin_len):

        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value,
            verbose=0
        )

        sampled_token_index = np.argmax(
            output_tokens[0, -1, :]
        )

        sampled_word = reverse_hin_index.get(
            sampled_token_index,
            ''
        )

        if sampled_token_index == end_token:
            break

        if sampled_word not in ['', '<start>']:
            decoded_sentence.append(sampled_word)

        target_seq = np.array([
            [sampled_token_index]
        ])

        states_value = [h, c]

    return ' '.join(decoded_sentence)   

In [ ]:
text = "how are you"

translation = translate_sentence(text)

print("English:", text)
print("Hindi:", translation)